In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
canvas_height = 512
canvas_width = 512
canvas = np.zeros((canvas_height, canvas_width, 3), dtype=np.uint8)

colors = {
    'B': (255, 0, 0),
    'G': (0, 255, 0),
    'R': (0, 0, 255)
}

current_color_key = 'B'
triangles = []

In [7]:
def triangle_points(x, y, size=35):
  return np.array([
      [x, y - size],
      [x - size, y + size],
      [x + size, y + size]
  ], dtype=np.int32)

def draw_triangle_callback(event, x, y, flags=None, param=None):
  global canvas, triangles
  if event == cv2.EVENT_LBUTTONDOWN:
    points = triangle_points(x, y, size=35)
    cv2.fillPoly(canvas, [points], colors[current_color_key])
    triangles.append({
        'center': (x, y),
        'points': points.tolist(),
        'color_key': colors[current_color_key]
    })

In [11]:
x_slider = widgets.IntSlider(value=256, min=35, max=canvas_width-36, step=1, description='X:')
y_slider = widgets.IntSlider(value=256, min=35, max=canvas_height-36, step=1, description='Y:')
size_slider = widgets.IntSlider(value=35, min=10, max=80, step=5, description='Size:')

button_b = widgets.Button(description='B - Blue', button_style='info')
button_g = widgets.Button(description='G - Green', button_style='success')
button_r = widgets.Button(description='R - Red', button_style='danger')
add_button = widgets.Button(description='Add Triangle', button_style='primary')
reset_button = widgets.Button(description='Reset Canvas')

status = widgets.HTML()
output_area = widgets.Output()

def refresh_display():
  with output_area:
    output_area.clear_output(wait=True)
    plt.figure(figsize=(7, 7))
    plt.imshow(canvas[..., ::-1])
    plt.title(f'Current color: {current_color_key} | Triangles: {len(triangles)}')
    plt.xlim(0, canvas_width)
    plt.ylim(canvas_height, 0)
    plt.grid(alpha=0.25)
    plt.show()
  status.value = f'<b>Current color:</b> {current_color_key} &nbsp; <b>Triangles:</b> {len(triangles)}'

def choose_color(key):
  global current_color_key
  current_color_key = color_key
  refresh_display()

def on_blue(button):
  choose_color('B')

def on_green(button):
  choose_color('G')

def on_red(button):
  choose_color('R')

def on_add_triangle(button):
  draw_triangle_callback(cv2.EVENT_LBUTTONDOWN, x_slider.value, y_slider.value)
  refresh_display()

def on_reset(button):
  global canvas, triangles
  canvas = np.zeros((canvas_height, canvas_width, 3), dtype=np.uint8)
  triangles = []
  refresh_display()

button_b.on_click(on_blue)
button_g.on_click(on_green)
button_r.on_click(on_red)
add_button.on_click(on_add_triangle)
reset_button.on_click(on_reset)

display(widgets.VBox([
    widgets.HBox([button_b, button_g, button_r]),
    x_slider,
    y_slider,
    size_slider,
    widgets.HBox([add_button, reset_button]),
    output_area,
    status
]))
refresh_display()

In [13]:
print(f'Number of triangles: {len(triangles)}')

for index, triangle in enumerate(triangles, start=1):
  print(
      f"Triangle {index}: center={triangle['center']}, "
      f"color={triangle['color_key']}"
  )

Number of triangles: 1
Triangle 1: center=(256, 256), color=(255, 0, 0)
